# Harmonisation 2025 / 2026

Ce notebook compare les réponses possibles des colonnes utilisées par les axes, pour écrire `HARMONISATION.md`.

On regarde chaque colonne avec `value_counts()` : si les réponses sont écrites pareil les deux années, on peut comparer ; sinon il faut une correspondance.

In [ ]:
import sys
sys.path.append('..')  # pour trouver le dossier analyse/ depuis notebooks/

import pandas as pd
from analyse.chargement import charger
from analyse.population import ajouter_zone, EN_EMPLOI

In [ ]:
d25 = charger(2025)
d26 = charger(2026)
print('2025 :', d25.shape, '| 2026 :', d26.shape)

## 1. Quelles colonnes existent les deux années ?

Je remarque : certaines colonnes ont changé de nom, d'autres n'existent qu'en 2026.

In [ ]:
colonnes = ['MainBranch', 'CodeMain', 'Employment', 'Country', 'WorkExp', 'JobSat',
            'RemoteWork', 'WorkSetting', 'OrgSize', 'ICorPM', 'Industry',
            'JobSatComplacent', 'JobEase', 'JobDifficulty', 'LearnAI', 'TechBudget']
pd.DataFrame({
    'en 2025': [c in d25.columns for c in colonnes],
    'en 2026': [c in d26.columns for c in colonnes],
}, index=colonnes)

## 2. Une fonction pour comparer les réponses des deux années

Elle met côte à côte le nombre de réponses de chaque modalité. Une case vide (`NaN`) veut dire que la réponse n'existe pas cette année-là.

In [ ]:
def comparer(col_2025, col_2026=None):
    col_2026 = col_2026 or col_2025
    return pd.DataFrame({
        '2025': d25[col_2025].value_counts(),
        '2026': d26[col_2026].value_counts(),
    })

## 3. Colonnes comparables telles quelles

In [ ]:
comparer('Employment')

In [ ]:
comparer('JobSat').sort_index()

In [ ]:
comparer('OrgSize')

In [ ]:
# RemoteWork (2025) est devenu WorkSetting (2026) : mêmes 5 réponses
comparer('RemoteWork', 'WorkSetting')

In [ ]:
# WorkExp : même question, en nombre d'années
pd.DataFrame({'2025': d25['WorkExp'].describe(), '2026': d26['WorkExp'].describe()})

In [ ]:
# les pays de nos zones s'écrivent-ils pareil ?
pays = ['France', 'Switzerland', 'Germany', 'Belgium', 'Luxembourg', 'Italy', 'Spain']
pd.DataFrame({'2025': d25['Country'].value_counts()[pays], '2026': d26['Country'].value_counts()[pays]})

## 4. Colonnes qui demandent une correspondance

### Population : `MainBranch` (2025) et `CodeMain` (2026)

Je remarque : la question a changé, il n'y a aucune réponse en commun.

In [ ]:
print(d25['MainBranch'].value_counts(), '\n')
print(d26['CodeMain'].value_counts())

In [ ]:
# correspondance choisie : voir HARMONISATION.md
CODENT_2025 = [
    'I am a developer by profession',
    'I am not primarily a developer, but I write code sometimes as part of my work/studies',
]
pop25 = d25[d25['MainBranch'].isin(CODENT_2025) & d25['Employment'].isin(EN_EMPLOI)]
pop25 = ajouter_zone(pop25)
print('Population 2025 :', len(pop25))
pop25['Zone'].value_counts()

### Rôle : `ICorPM`

2 réponses en 2025, 4 en 2026 : il faut regrouper.

In [ ]:
comparer('ICorPM')

### Secteur : `Industry`

Je remarque : la catégorie ESN (`IT services, consulting...`) n'existe qu'en 2026, et `Software Development` est beaucoup plus gros en 2025.

In [ ]:
comparer('Industry')

## 5. Ce que je retiens

- Comparables : `Employment`, `Country`, `WorkExp`, `JobSat`, `OrgSize`, `RemoteWork` → `WorkSetting`.
- Avec correspondance : `MainBranch` → `CodeMain`, `ICorPM` en deux groupes.
- Pas de comparaison ESN avec 2025.
- Seulement en 2026 : `JobSatComplacent`, `JobEase`, `JobDifficulty`, `LearnAI`, `TechBudget`.

Le détail est dans `HARMONISATION.md`.